In [ ]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix, classification_report
)

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

from xgboost import XGBClassifier

In [73]:
diabetes = pd.read_csv("C:/Documants/mediscan/MediScan/Datasets/diabetes.csv")
diabetes.info()

<class 'pandas.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   768 non-null    int64  
 2   BloodPressure             768 non-null    int64  
 3   SkinThickness             768 non-null    int64  
 4   Insulin                   768 non-null    int64  
 5   BMI                       768 non-null    float64
 6   DiabetesPedigreeFunction  768 non-null    float64
 7   Age                       768 non-null    int64  
 8   Outcome                   768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB


In [74]:
diabetes.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [75]:
(diabetes == 0).sum()

Pregnancies                 111
Glucose                       5
BloodPressure                35
SkinThickness               227
Insulin                     374
BMI                          11
DiabetesPedigreeFunction      0
Age                           0
Outcome                     500
dtype: int64

In [76]:
diabetes.duplicated().sum()

np.int64(0)

In [77]:
# =========================================================
# FEATURE SELECTION
# =========================================================
X = diabetes.drop('Outcome' , axis = 1) ## selecting all columns except outcome
y = diabetes['Outcome']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [79]:

# =========================================================
# PREPROCESSING STEP: ZERO -> NaN -> MEDIAN IMPUTE
# (fit medians on train only, apply same values to test)
# =========================================================
zero_as_missing = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
X_train = X_train.copy()
X_test = X_test.copy()

X_train[zero_as_missing] = X_train[zero_as_missing].replace(0, np.nan)
X_test[zero_as_missing] = X_test[zero_as_missing].replace(0, np.nan)

impute_medians = {}
for col in zero_as_missing:
    median_val = X_train[col].median()
    impute_medians[col] = median_val
    X_train[col] = X_train[col].fillna(median_val)
    X_test[col] = X_test[col].fillna(median_val)

print("\nImputation medians (from train set):", impute_medians)


Imputation medians (from train set): {'Glucose': np.float64(117.0), 'BloodPressure': np.float64(72.0), 'SkinThickness': np.float64(29.0), 'Insulin': np.float64(125.0), 'BMI': np.float64(32.4)}


In [80]:
# =========================================================
# GRIDSEARCHCV WITH SMOTE INSIDE THE PIPELINE
# =========================================================

# --- Random Forest ---
rf_pipeline = ImbPipeline([
    ("smote", SMOTE(random_state=42)),
    ("rf", RandomForestClassifier(random_state=42, class_weight="balanced"))
])
rf_params = {
    "rf__n_estimators": [100, 200, 300],
    "rf__max_depth": [None, 5, 10, 15],
    "rf__min_samples_split": [2, 5, 10],
    "rf__min_samples_leaf": [1, 2, 4]
}
rf_grid = GridSearchCV(rf_pipeline, rf_params, cv=5, scoring="accuracy", n_jobs=-1)
rf_grid.fit(X_train, y_train)

# --- Gradient Boosting ---
gb_pipeline = ImbPipeline([
    ("smote", SMOTE(random_state=42)),
    ("gb", GradientBoostingClassifier(random_state=42))
])
gb_params = {
    "gb__n_estimators": [100, 200, 300],
    "gb__learning_rate": [0.01, 0.05, 0.1],
    "gb__max_depth": [2, 3, 4],
    "gb__min_samples_split": [2, 5, 10]
}
gb_grid = GridSearchCV(gb_pipeline, gb_params, cv=5, scoring="accuracy", n_jobs=-1)
gb_grid.fit(X_train, y_train)

# --- Logistic Regression ---
lr_pipeline = ImbPipeline([
    ("scaler", StandardScaler()),
    ("smote", SMOTE(random_state=42)),
    ("lr", LogisticRegression(max_iter=1000, random_state=42, class_weight="balanced"))
])
lr_params = {
    "lr__C": [0.01, 0.1, 1, 10, 100]
}
lr_grid = GridSearchCV(lr_pipeline, lr_params, cv=5, scoring="accuracy", n_jobs=-1)
lr_grid.fit(X_train, y_train)

# =========================================================
# 11. REPORT CV RESULTS (the honest, leakage-free numbers)
# =========================================================
print("==============================")
print("Best CV Scores (leakage-free)")
print("==============================")
print(f"Random Forest:       {rf_grid.best_score_:.4f}  | {rf_grid.best_params_}")
print(f"Gradient Boosting:   {gb_grid.best_score_:.4f}  | {gb_grid.best_params_}")
print(f"Logistic Regression: {lr_grid.best_score_:.4f}  | {lr_grid.best_params_}")

KeyboardInterrupt: 

In [ ]:
# =========================================================
# EVALUATE BEST SMOTE MODELS ON HELD-OUT TEST SET
# =========================================================
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

for name, grid in [("Random Forest", rf_grid), ("Gradient Boosting", gb_grid), ("Logistic Regression", lr_grid)]:
    best_model = grid.best_estimator_
    y_pred = best_model.predict(X_test)

    print(f"\n{'='*50}")
    print(f"{name} — Test Set Results")
    print(f"{'='*50}")
    print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
    print(classification_report(y_test, y_pred))
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))


Random Forest — Test Set Results
Accuracy: 0.7532
              precision    recall  f1-score   support

           0       0.84      0.76      0.80       100
           1       0.62      0.74      0.68        54

    accuracy                           0.75       154
   macro avg       0.73      0.75      0.74       154
weighted avg       0.77      0.75      0.76       154

Confusion Matrix:
[[76 24]
 [14 40]]

Gradient Boosting — Test Set Results
Accuracy: 0.7597
              precision    recall  f1-score   support

           0       0.86      0.75      0.80       100
           1       0.63      0.78      0.69        54

    accuracy                           0.76       154
   macro avg       0.74      0.76      0.75       154
weighted avg       0.78      0.76      0.76       154

Confusion Matrix:
[[75 25]
 [12 42]]

Logistic Regression — Test Set Results
Accuracy: 0.7338
              precision    recall  f1-score   support

           0       0.83      0.74      0.78       100
